In [1]:
import sys
from pathlib import Path
import json
import torch

from app.training.formatter import DatasetFormatter
from app.schemas import TrainingProfile, LoRAProfile
from app.training.lora_config import LoRAConfigFactory
from app.training.model_loader import ModelLoader
from app.training.trainer import DomainTrainer

/Users/macstudio/Desktop/Development/DomainForge V1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks" or not (BASE_DIR / "app").exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"Project Base Directory: {BASE_DIR}")

Project Base Directory: /Users/macstudio/Desktop/Development/DomainForge V1


In [3]:
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
FINAL_DATA_DIR = BASE_DIR / "data" / "final"
QA_DATASET_FILE = PROCESSED_DATA_DIR / "qa_dataset.jsonl"

assert QA_DATASET_FILE.exists(), f"ERROR: The validated data file could not be found: {QA_DATASET_FILE}"

with open(QA_DATASET_FILE, "r", encoding="utf-8") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

print(f"Total number of verified records: {len(raw_lines)}")

Total number of verified records: 1


In [4]:
formatter = DatasetFormatter()
summary = formatter.process_and_export(
    input_qa_path=QA_DATASET_FILE,
    output_dir=FINAL_DATA_DIR,
    train_ratio=0.8,
    seed=42
)

print(f"Total Number of Records : {summary.total_count}")
print(f"Train Set               : {summary.train_count} value -> {summary.train_path}")
print(f"Eval Set                : {summary.eval_count} value -> {summary.eval_path}")

Total Number of Records : 1
Train Set               : 1 value -> /Users/macstudio/Desktop/Development/DomainForge V1/data/final/train.jsonl
Eval Set                : 0 value -> /Users/macstudio/Desktop/Development/DomainForge V1/data/final/eval.jsonl


In [5]:
with open(summary.train_path, "r", encoding="utf-8") as f:
    sample_train = json.loads(f.readline())

print("=== SAMPLE TRAINING RECORD (ChatML Format) ===")
print(json.dumps(sample_train, indent=2, ensure_ascii=False))

=== SAMPLE TRAINING RECORD (ChatML Format) ===
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert AI Governance Advisor. Provide rigorous, balanced, and technically grounded answers based on international AI governance frameworks and technical standards."
    },
    {
      "role": "user",
      "content": "What are the key differences in the characteristics of trustworthy AI systems as outlined in NIST.AI.100-1 and NIST.AI.600-1 frameworks?"
    },
    {
      "role": "assistant",
      "content": "The NIST.AI.100-1 framework emphasizes characteristics such as valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed. In contrast, the NIST.AI.600-1 framework lists similar characteristics but organizes them differently, focusing on accountable and transparent, explainable and interpretable, fair with harmful bias managed, privacy enhanced, safe, secure

In [7]:
print(" PHASE 8 DRY-RUN & SMOKE TEST ")

loader = ModelLoader("Qwen/Qwen2.5-3B-Instruct")
print(f"Target Device Detection : {loader.device}")

lora_prof = LoRAProfile(r=16, lora_alpha=32)
peft_cfg = LoRAConfigFactory.create(lora_prof)
print(f"LoRA Configuration : r={peft_cfg.r}, alpha={peft_cfg.lora_alpha}")
print(f"Target Layers      : {peft_cfg.target_modules}")

train_prof = TrainingProfile(
    base_model_id="Qwen/Qwen2.5-3B-Instruct",
    num_train_epochs=3,
    per_device_train_batch_size=2
)
trainer_engine = DomainTrainer(train_profile=train_prof, lora_profile=lora_prof)
print("The DomainTrainer object has been successfully constructed (fine-tuning has not been triggered).")

 PHASE 8 DRY-RUN & SMOKE TEST 
Target Device Detection : mps
LoRA Configuration : r=16, alpha=32
Target Layers      : {'v_proj', 'gate_proj', 'down_proj', 'q_proj', 'k_proj', 'up_proj', 'o_proj'}
The DomainTrainer object has been successfully constructed (fine-tuning has not been triggered).
